### P2.3 — Deep Learning Foundations | Python to Gen AI Course
### P2.3.5 — Overfitting, Underfitting and How to Fix Them

---
## 🔁 Quick Recap — Where We Are

In P2.3.4 we saw how a network learns:

```
Forward Pass → Loss → Backprop → Weight Update → Repeat
```

And we watched the training loss drop nicely over epochs.

But training loss dropping is not enough.  
The real question is: **does the model work on data it has never seen before?**

That is where overfitting and underfitting come in —  
and understanding them is what separates a model that works in a notebook  
from one that actually works in the real world.

---
## 🎯 The Real Goal — Generalisation

A model that only works on training data is useless.  
The real goal is **generalisation** — performing well on new, unseen data.

```
Train a spam detector on 10,000 emails
     ↓
It must correctly classify NEW emails it has never seen
     ↓
If it only works on the 10,000 training emails → useless in production
```

To measure generalisation, we always split data into **Train** and **Test** sets —  
exactly as we did in the ML module.

```
Train Set  →  model learns from this
Test Set   →  model is evaluated on this (never seen during training)
```

We then compare **Train Loss vs Test Loss** to diagnose what went wrong.

<img src="generalisation.png" width="700"/>
<!-- Visual: two boxes. LEFT: 'Training Data' with brain icon — 'model learns here'. RIGHT: 'Test Data (unseen)' with question mark — 'model is judged here'. Arrow from left to right. Below: 'Goal: perform well on the right box'. -->

---
## ❌ Overfitting — Memorising Instead of Learning

Overfitting happens when the model learns the training data **too well** —  
including all the noise and random quirks that don't generalise.

```
Train Loss  →  very low   (model nailed the training data)
Test Loss   →  very high  (model fails on new data)

Big gap between train and test = Overfitting
```

**Real world analogy:**  
A student who memorises every past exam paper word by word.  
They score 100% on practice papers — but fail when the actual exam has new questions.  
They memorised the answers. They did not learn the concept.

```
In DL:
  Train Accuracy: 99%
  Test  Accuracy: 62%   ← huge gap — overfitting
```

**Common causes:**
```
Too many neurons/layers  →  network too large for the data
Too many epochs          →  trained for too long
Too little data          →  not enough examples to generalise from
```

<img src="overfitting.png" width="750"/>
<!-- Visual: two graphs side by side. LEFT: scatter plot with a wildly curving line threading through every single training point perfectly — labeled 'Overfit — memorised every point'. RIGHT: smooth curve that captures the trend but misses some points — labeled 'Good Fit — learned the pattern'. -->

---
## ❌ Underfitting — Not Learning Enough

Underfitting is the opposite — the model is too simple to capture the pattern.

```
Train Loss  →  high   (model could not even learn training data)
Test Loss   →  high   (obviously fails on new data too)

Both scores low = Underfitting
```

**Real world analogy:**  
A student who only studied for one hour before the exam.  
They don't know the material — they fail on everything.

```
In DL:
  Train Accuracy: 55%
  Test  Accuracy: 52%   ← both low — underfitting
```

**Common causes:**
```
Too few neurons/layers  →  network too small
Too few epochs          →  did not train long enough
Learning rate too high  →  never converged properly
```

<img src="underfitting.png" width="750"/>
<!-- Visual: scatter plot with a nearly flat straight line through curved data — labeled 'Underfit — too simple to capture the curve'. Beside it: the good fit curve for comparison. -->

---
## ✅ Good Fit — The Goal

```
Train Accuracy: 95%
Test  Accuracy: 92%   ← small gap — model generalised well
```

The gap between train and test will never be zero — that is completely normal.  
A small gap means the model learned the real pattern, not just the training data.

```
─────────────────────────────────────────────────────────────────
 Scenario        Train Score    Test Score    Diagnosis
─────────────────────────────────────────────────────────────────
 Good Fit        High           High          ✅ Generalises well
 Overfitting     Very High      Low           ❌ Memorised training
 Underfitting    Low            Low           ❌ Too simple
─────────────────────────────────────────────────────────────────
```

<img src="fit_comparison.png" width="800"/>
<!-- Visual: 3 loss curve pairs side by side. Each has Train curve (blue) and Test curve (orange). LEFT (Good Fit): both drop together, small gap. MIDDLE (Overfit): train drops very low, test rises — big gap. RIGHT (Underfit): both stay high, neither drops. Labels under each. -->

---
## 🛠️ Fix 1 — Dropout

During training, **randomly switch off a fraction of neurons** in each forward pass.

```
Normal pass:   Neuron 1 ✓  Neuron 2 ✓  Neuron 3 ✓  Neuron 4 ✓
Dropout pass:  Neuron 1 ✓  Neuron 2 ✗  Neuron 3 ✓  Neuron 4 ✗
                                ↑                        ↑
                           switched off             switched off
```

Why does this help?  
The network can no longer rely on specific neurons to memorise specific answers.  
It is forced to learn more robust, distributed patterns.

```python
# Keras — add Dropout after a hidden layer
layers.Dense(64, activation='relu'),
layers.Dropout(0.3),   # 30% of neurons switched off randomly during training
```

> Dropout is only active during **training.**  
> During inference (prediction), all neurons are active — nothing is dropped.

<img src="dropout.png" width="700"/>
<!-- Visual: two network diagrams. LEFT: full network all neurons active — 'Normal Pass'. RIGHT: same network with some neurons grayed out and X marks — 'Dropout Pass — 30% randomly off'. -->

---
## 🛠️ Fix 2 — Early Stopping

Stop training the moment **test loss stops improving** — even if train loss is still dropping.

```
Epoch 50:   Train Loss 0.12  Test Loss 0.15  ← still improving
Epoch 80:   Train Loss 0.08  Test Loss 0.16  ← test loss rising slightly
Epoch 100:  Train Loss 0.05  Test Loss 0.21  ← overfitting is happening
                                    ↑
                     Stop here at epoch 50 — that was the best
```

```python
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',        # Watch test (validation) loss
    patience=10,               # Stop if no improvement for 10 epochs
    restore_best_weights=True  # Rewind to the best epoch's weights
)

model.fit(X_train, y_train,
          validation_data=(X_test, y_test),
          epochs=500,
          callbacks=[early_stop])
```

<img src="early_stopping.png" width="700"/>
<!-- Visual: line graph with Train Loss (blue, keeps dropping) and Test Loss (orange, drops then rises). Vertical dashed line where test loss starts rising — labeled 'Stop Here — Early Stopping'. Area to the right shaded red labeled 'Overfitting Zone'. -->

---
## 🛠️ Fix 3 — More Data and Simpler Network

### More Data
```
More diverse training examples → harder to memorise → forced to generalise
```
The simplest fix — if the model is memorising, give it more variety to learn from.

### Simpler Network
```
Fewer neurons / fewer layers → fewer parameters → less capacity to memorise

Too large for your data → overfit
Reduce size             → better generalisation
```

---
## 💻 Let's See It — Overfitting vs Dropout in Code

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

# ── DATA ──────────────────────────────────────────────────────────
X = np.array([
    [1000, 2, 10], [1200, 3,  5], [1500, 3,  8],
    [1800, 4,  3], [2000, 4,  2], [2300, 5,  1],
    [1100, 2,  7], [1400, 3,  6], [1700, 4,  4],
    [1900, 4,  3], [2100, 5,  2], [2500, 5,  1],
], dtype=float)

y = np.array([200, 250, 300, 360, 400, 450,
              220, 280, 330, 380, 420, 470], dtype=float)

# ── PHASE 1 : SPLIT ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

In [ ]:
# ── MODEL A — No Dropout ──────────────────────────────────────────
# Large network on small data — likely to overfit
model_a = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(3,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])
model_a.compile(optimizer='adam', loss='mse')

history_a = model_a.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=300, verbose=0
)
print("Model A (No Dropout) — training done")

In [ ]:
# ── MODEL B — Dropout + Early Stopping ───────────────────────────
model_b = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(3,)),
    layers.Dropout(0.3),   # 30% neurons off during training
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1)
])
model_b.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history_b = model_b.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=300, callbacks=[early_stop], verbose=0
)
print(f"Model B — stopped at epoch {len(history_b.history['loss'])}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 3 : EVALUATE — Compare both models
# ══════════════════════════════════════════════════════════════════
def evaluate_model(name, history):
    train_loss = history.history['loss'][-1]
    test_loss  = history.history['val_loss'][-1]
    gap        = train_loss - test_loss
    print(f"\n{name}")
    print(f"  Train Loss : {train_loss:.2f}")
    print(f"  Test  Loss : {test_loss:.2f}")
    if abs(gap) < 20:
        print("  ✅ Good Fit")
    elif train_loss < test_loss - 20:
        print("  ⚠️  Overfitting — train much better than test")
    else:
        print("  ⚠️  Underfitting — both scores are poor")

print("=== PHASE 3 : EVALUATE ===")
evaluate_model("Model A — No Dropout",               history_a)
evaluate_model("Model B — Dropout + Early Stopping", history_b)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 4 : INFERENCE — Compare predictions
# ══════════════════════════════════════════════════════════════════
new_house = np.array([[1600, 3, 4]])

pred_a = model_a.predict(new_house, verbose=0)[0][0]
pred_b = model_b.predict(new_house, verbose=0)[0][0]

print("=== PHASE 4 : INFERENCE ===")
print(f"House: 1600 sqft, 3 bed, 4 yrs old")
print(f"Model A (No Dropout)  → ₹{pred_a:.1f}K")
print(f"Model B (With Dropout)→ ₹{pred_b:.1f}K")

---
## ✅ Summary — What You Learned

| Concept | Key Point |
|---|---|
| Generalisation | Real goal — perform well on unseen data, not just training data |
| Overfitting | Train high, test low — model memorised training data |
| Underfitting | Both low — model too simple to learn the pattern |
| Good Fit | Both high with small gap — learned the real pattern |
| Dropout | Randomly switch off neurons during training — prevents memorisation |
| Early Stopping | Stop when test loss stops improving — avoids overtraining |
| More Data | More diverse examples → harder to memorise → better generalisation |
| Simpler Network | Fewer parameters → less capacity to overfit |

---

**👉 Next: P2.3.6 — TensorFlow, Keras and PyTorch — Understanding the Landscape**